In [1]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [2]:
texts = [
    "I love this film",
    "Extraordinary cast",
    "Very good acting",
    "Excellent Story",
    "I hate this movie",
    "Terrible",
    "bad story",
    "disgusting"
]

#Lables
#1=positive
#0=negative
labels = np.array([1, 1, 1, 1, 0, 0, 0, 0])

In [3]:
vocab_size = 1000
max_length = 6

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

X_train = pad_sequences(sequences,maxlen=max_length,padding="post")

print("Word Index = ")
print(tokenizer.word_index)

print("\nInput Sequence - ")
print(X_train)

Word Index = 
{'<OOV>': 1, 'i': 2, 'this': 3, 'story': 4, 'love': 5, 'film': 6, 'extraordinary': 7, 'cast': 8, 'very': 9, 'good': 10, 'acting': 11, 'excellent': 12, 'hate': 13, 'movie': 14, 'terrible': 15, 'bad': 16, 'disgusting': 17}

Input Sequence - 
[[ 2  5  3  6  0  0]
 [ 7  8  0  0  0  0]
 [ 9 10 11  0  0  0]
 [12  4  0  0  0  0]
 [ 2 13  3 14  0  0]
 [15  0  0  0  0  0]
 [16  4  0  0  0  0]
 [17  0  0  0  0  0]]


In [4]:
# Positional Embeddings in Transformers

class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )
 
        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )
 
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb

In [5]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )
 
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
 
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
 
    def call(self, inputs):
        # Self-attention (Query, Key, Value are calculated here using the same input)
        # The attention layer takes the input and computes
        # attention scores to capture relationships between different positions in the sequence.
        attention_output = self.attention(inputs, inputs)
 
        # Add + Normalize
        out1 = self.layernorm1(inputs + attention_output)
 
        # Feed-forward network
        ffn_output = self.ffn(out1)
 
        # Add + Normalize
        out2 = self.layernorm2(out1 + ffn_output)
 
        return out2

In [10]:
# build the model
embed_dim = 32
num_heads = 4
ff_dim = 32
inputs = layers.Input(shape=(max_length,))
 
x = TokenAndPositionEmbedding(
    max_length=max_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(inputs)
 
x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)
 
x = layers.GlobalAveragePooling1D()(x)
 
outputs = layers.Dense(1, activation="sigmoid")(x)
 
model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [11]:
model.compile(

    optimizer = "adam",
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_1  │ (None, 6, 32)          │        32,192 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 6, 32)          │        19,040 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51,265 (200.25 KB)

 Trainable params: 51,265 (200.25 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.fit(X_train, labels, epochs=30, batch_size=2, verbose=1)

Epoch 1/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.5000 - loss: 1.3053
Epoch 2/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2500 - loss: 1.0098     
Epoch 3/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5000 - loss: 0.8737 
Epoch 4/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6250 - loss: 0.6121 
Epoch 5/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7500 - loss: 0.5291 
Epoch 6/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7500 - loss: 0.5362 
Epoch 7/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7500 - loss: 0.4882 
Epoch 8/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.4193 
Epoch 9/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.3872 
Epoch 10/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.3536 
Epoch 11/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.3280 
Epoch 12/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.2845

In [13]:
Test_sentences = [
    "I love the film",
    "This movie was awful",
    "Too bad",
    "Holy mid",
    "Boring but good"
]
 
test_seq = tokenizer.texts_to_sequences(Test_sentences)
test_pad = pad_sequences(test_seq, maxlen=max_length, padding="post")
predictions = model.predict(test_pad)
 
for sentence, prediction in zip(Test_sentences, predictions):
    print(sentence, "->", prediction[0])
    if prediction[0] > 0.5:
        print("Prediction: Positive")
    else:
        print("Prediction: Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
I love the film -> 0.9926571
Prediction: Positive
This movie was awful -> 0.33056805
Prediction: Negative
Too bad -> 0.021593047
Prediction: Negative
Holy mid -> 0.36695862
Prediction: Negative
Boring but good -> 0.9288193
Prediction: Positive
